# JSONL Store
> Append-only JSONL session store

In [ ]:
#| default_exp store.jsonl

In [ ]:
#| export
import orjson
from pathlib import Path
from typing import Optional, List, Dict
from dataclasses import asdict
from memexcode.core import gen_id, MessageEntry, CompactionEntry, SessionHeader
from memexcode.session import SessionEntry
from memexcode.pluginspec import SessionABC, SessionStore, SessionInfo, TreeNode, hookimpl

ImportError: cannot import name 'SessionEntry' from partially initialized module 'memexcode.session' (most likely due to a circular import) (/app/data/dialogs/b10Xp6TI7aVVLp8GyipIkw/memexcode/memexcode/session.py)

In [ ]:
#| export
class JsonlSession(SessionABC):
    def __init__(self, file_path: Path, header_id: str):
        self._file = Path(file_path)
        self._header_id = header_id
        self._entries: List[SessionEntry] = []
        self._by_id: Dict[str, SessionEntry] = {}
        self._load()

    def _load(self):
        if not self._file.exists():
            return
        with open(self._file, 'rb') as f:
            first = True
            for line in f:
                if not line.strip():
                    continue
                if first:
                    first = False
                    continue  # skip header
                entry = SessionEntry.from_dict(orjson.loads(line))
                self._entries.append(entry)
                self._by_id[entry.id] = entry

    async def append(self, entry: SessionEntry) -> None:
        if not entry.id:
            entry._entry.id = gen_id()
        if self._entries:
            entry._entry.parentId = self._entries[-1].id
        else:
            entry._entry.parentId = self._header_id
        self._entries.append(entry)
        self._by_id[entry.id] = entry
        with open(self._file, 'ab') as f:
            f.write(orjson.dumps(entry.to_dict()) + b'\n')

    async def get_by_id(self, entry_id: str) -> Optional[SessionEntry]:
        return self._by_id.get(entry_id)

    async def get_path_to_root(self, leaf_id: str) -> List[SessionEntry]:
        path = []
        current = self._by_id.get(leaf_id)
        while current:
            path.insert(0, current)
            current = self._by_id.get(current.parentId) if current.parentId else None
        return path

    async def load_all(self) -> List[SessionEntry]:
        return list(self._entries)

    async def compact(self, first_kept_id: str, summary: str) -> None:
        idx = next((i for i, e in enumerate(self._entries) if e.id == first_kept_id), None)
        if idx is None:
            raise ValueError(f'first_kept_id {first_kept_id} not found')
        parent = self._entries[idx].parentId if idx > 0 else None
        compact = SessionEntry(CompactionEntry(
            parentId=parent, summary=summary, firstKeptEntryId=first_kept_id
        ))
        self._entries = [compact] + self._entries[idx:]
        self._by_id = {e.id: e for e in self._entries}
        with open(self._file, 'wb') as f:
            for e in self._entries:
                f.write(orjson.dumps(e.to_dict()) + b'\n')

    async def get_tree(self, root_id: Optional[str] = None) -> TreeNode:
        root = self._entries[0] if not root_id else self._by_id.get(root_id)
        if not root:
            return TreeNode(SessionEntry(MessageEntry()), [])
        children = {e.id: [] for e in self._entries}
        for e in self._entries:
            if e.parentId and e.parentId in children:
                children[e.parentId].append(e)
        def build(entry):
            return TreeNode(entry, [build(c) for c in children.get(entry.id, [])])
        return build(root)

class JsonlStore(SessionStore):
    def __init__(self, root_dir: Path):
        self.root = Path(root_dir)
        self.root.mkdir(parents=True, exist_ok=True)

    def get_session(self, session_id: str) -> SessionABC:
        file_path = self.root / f'{session_id}.jsonl'
        header_id = session_id
        if file_path.exists():
            with open(file_path, 'rb') as f:
                first = f.readline()
                if first:
                    header = orjson.loads(first)
                    header_id = header.get('id', session_id)
        return JsonlSession(file_path, header_id)

    async def create(self, cwd: str, name: Optional[str] = None, parent_session: Optional[str] = None) -> SessionInfo:
        info = SessionInfo(id=gen_id(), cwd=cwd, created_at=now_iso(), updated_at=now_iso(),
                          name=name, parent_session=parent_session)
        file_path = self.root / f'{info.id}.jsonl'
        header = SessionHeader(id=info.id, cwd=cwd, parentSession=parent_session)
        with open(file_path, 'wb') as f:
            f.write(orjson.dumps(asdict(header)) + b'\n')
        return info

    async def list(self) -> List[SessionInfo]:
        infos = []
        for f in sorted(self.root.glob('*.jsonl')):
            with open(f, 'rb') as fh:
                first = fh.readline()
                if first:
                    header = orjson.loads(first)
                    infos.append(SessionInfo(
                        id=header.get('id', f.stem),
                        cwd=header.get('cwd', ''),
                        created_at=header.get('timestamp', ''),
                        updated_at=header.get('timestamp', ''),
                        parent_session=header.get('parentSession')
                    ))
        return infos

    async def rename(self, session_id: str, name: str) -> None:
        pass

    async def delete(self, session_id: str) -> None:
        file_path = self.root / f'{session_id}.jsonl'
        if file_path.exists():
            file_path.unlink()

    async def fork(self, session_id: str, leaf_id: Optional[str] = None, name: Optional[str] = None) -> SessionInfo:
        old = self.get_session(session_id)
        path = await old.get_path_to_root(leaf_id) if leaf_id else await old.load_all()
        old_info = next((i for i in await self.list() if i.id == session_id), None)
        info = await self.create(cwd=old_info.cwd if old_info else '', name=name, parent_session=session_id)
        new = self.get_session(info.id)
        for entry in path:
            await new.append(entry)
        return info

@hookimpl
def get_store():
    return ('jsonl', JsonlStore)

NameError: name 'SessionABC' is not defined